In [1]:
!pip install krippendorff datasets pandas --break-system-packages

In [2]:
"""
Inter-judge agreement (Krippendorff's alpha) + judge-score aggregation
for LLM-as-judge pipeline outputs.

Install deps:
    pip install krippendorff datasets pandas --break-system-packages

Expected input shape (one row per item_id x judge_model):
    item_id, system, judge_model, timestamp, status, raw_response,
    parsed_json, error, recomputed_formatting_checks_passed,
    recomputed_formatting_score, recomputed_readability_ratio,
    recomputed_readability_score, validation_issues, is_consistent
"""

import numpy as np
import os
import pandas as pd
import krippendorff
from datasets import load_dataset

# --------------------------------------------------------------------------
# 1. CONFIG — edit these for your setup
# --------------------------------------------------------------------------

# Hugging Face dataset repo ids (swap in your actual repo names)
HF_DATASETS = {
    "exp10": "businessrules/exp10_promptA_results",
    "gpt4.1": "businessrules/gpt4.1_promptA_results",
}
HF_SPLIT = "train"

# Metrics to evaluate, and the correct Krippendorff level of measurement
# for each. This matters: alpha is computed differently depending on
# whether the scale is nominal, ordinal, interval, or ratio.
#   - *_score columns are ordinal (1-5 Likert-style buckets)
#   - *_checks_passed is a bounded count -> treat as interval (or ordinal
#     if you want to be conservative)
#   - *_ratio is a continuous 0-1 proportion -> ratio
METRIC_LEVELS = {
    "recomputed_formatting_score": "ordinal",
    "recomputed_readability_score": "ordinal",
    "recomputed_formatting_checks_passed": "interval",
    "recomputed_readability_ratio": "ratio",
}

ITEM_COL = "item_id"
JUDGE_COL = "judge_model"
STATUS_COL = "status"  # rows where this != "ok" are dropped before analysis


# --------------------------------------------------------------------------
# 2. LOADING
# --------------------------------------------------------------------------

def _null_columns_to_string(table: "pa.Table") -> "pa.Table":
    """
    Cast any Arrow "null"-typed columns to string.
 
    A column that is entirely None in one parquet shard gets inferred by
    PyArrow as type `null`. If another shard has real string values in
    that same column (e.g. `error`, which is empty for every successful
    judging call but populated for failures), concatenating/casting the
    two together raises `TypeError: Couldn't cast array of type string to
    null`. This is exactly the `datasets`-library error you hit. Casting
    null -> string up front avoids it entirely.
    """
    import pyarrow as pa
 
    new_cols = []
    for col, field in zip(table.columns, table.schema):
        if pa.types.is_null(field.type):
            col = col.cast(pa.string())
        new_cols.append(col)
    return pa.Table.from_arrays(new_cols, names=table.schema.names)
    
def load_judged_dataset(repo_id: str, split: str = HF_SPLIT) -> pd.DataFrame:
    """
    Load a judged dataset from the Hugging Face Hub.
 
    This reads the split's parquet shard(s) directly with
    `huggingface_hub` + `pyarrow` instead of going through
    `datasets.load_dataset()`. That sidesteps a `datasets`-library bug
    where a column that's entirely null in one shard (commonly `error`,
    since it's empty for every row where judging succeeded) fails to
    reconcile against another shard where the same column holds real
    strings -- exactly the `DatasetGenerationError` /
    `TypeError: Couldn't cast array of type string to null` you saw.
 
    Falls back to `datasets.load_dataset` only if `huggingface_hub` isn't
    available.
    """
    try:
        from huggingface_hub import HfApi, hf_hub_download
        import pyarrow.parquet as pq
    except ImportError:
        if load_dataset is None:
            raise ImportError(
                "Neither `huggingface_hub`+`pyarrow` nor `datasets` is "
                "available. Install one of them, or load your data "
                "manually into a DataFrame and pass it straight into "
                "analyze_dataset()."
            )
        ds = load_dataset(repo_id, split=split)
        return ds.to_pandas()
 
    api = HfApi()
    all_files = api.list_repo_files(repo_id, repo_type="dataset")
    parquet_files = [f for f in all_files if f.endswith(".parquet")]
 

    split_files = [f for f in parquet_files if split in f] or parquet_files
    if not split_files:
        raise FileNotFoundError(
            f"No parquet files found in dataset repo '{repo_id}'. "
            f"Files present: {all_files}"
        )
 
    frames = []
    for f in split_files:
        local_path = hf_hub_download(repo_id, f, repo_type="dataset")
        table = pq.read_table(local_path)
        table = _null_columns_to_string(table)
        frames.append(table.to_pandas())
 
    return pd.concat(frames, ignore_index=True, sort=False)
 
 
def clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    Basic hygiene before any stats:
      - keep only successfully-parsed judge responses
      - drop exact duplicate (item_id, judge_model) rows, keeping the latest
        by timestamp (pipeline retries can otherwise create dupes)
    """
    df = df.copy()
    if STATUS_COL in df.columns:
        df = df[df[STATUS_COL] == "ok"]
 
    if "timestamp" in df.columns:
        df = df.sort_values("timestamp")
 
    df = df.drop_duplicates(subset=[ITEM_COL, JUDGE_COL], keep="last")
    return df
 

# --------------------------------------------------------------------------
# 3. KRIPPENDORFF'S ALPHA
# --------------------------------------------------------------------------

def build_reliability_matrix(df: pd.DataFrame, metric: str) -> np.ndarray:
    """
    Reshape long-format (item_id, judge_model, metric) data into the
    (n_judges x n_items) matrix krippendorff.alpha expects.
    Missing judge/item combinations become NaN, which krippendorff
    handles natively as "no rating".
    """
    pivot = df.pivot_table(index=JUDGE_COL, columns=ITEM_COL, values=metric, aggfunc="first")
    return pivot.to_numpy(dtype=float)


def compute_alpha(df: pd.DataFrame, metric: str, level: str) -> float:
    matrix = build_reliability_matrix(df, metric)
    return krippendorff.alpha(reliability_data=matrix, level_of_measurement=level)


def compute_all_alphas(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for metric, level in METRIC_LEVELS.items():
        if metric not in df.columns:
            continue
        alpha = compute_alpha(df, metric, level)
        n_items = df[ITEM_COL].nunique()
        n_judges = df[JUDGE_COL].nunique()
        rows.append(
            {
                "metric": metric,
                "level_of_measurement": level,
                "krippendorff_alpha": alpha,
                "n_items": n_items,
                "n_judges": n_judges,
            }
        )
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# 4. AGGREGATING JUDGE SCORES PER ITEM
# --------------------------------------------------------------------------

def aggregate_scores(df: pd.DataFrame, agg_funcs=("mean", "median")) -> pd.DataFrame:
    """
    Collapse the 3 (or n) judge rows per item_id into a single row per item,
    with one aggregated column per metric per agg function, plus the raw
    per-judge spread (std) so you can see disagreement at a glance.
    """
    metrics = [m for m in METRIC_LEVELS if m in df.columns]

    agg_map = {m: list(agg_funcs) + ["std", "count"] for m in metrics}
    grouped = df.groupby(ITEM_COL).agg(agg_map)

    # flatten MultiIndex columns: (metric, func) -> metric_func
    grouped.columns = [f"{metric}_{func}" for metric, func in grouped.columns]
    grouped = grouped.reset_index()

    # keep "system" (finetuned/etc.) if it's constant per item
    if "system" in df.columns:
        system_map = df.groupby(ITEM_COL)["system"].first()
        grouped = grouped.merge(system_map, on=ITEM_COL, how="left")

    return grouped


# --------------------------------------------------------------------------
# 5. PUTTING IT TOGETHER FOR ONE DATASET
# --------------------------------------------------------------------------

def analyze_dataset(name: str, df: pd.DataFrame):
    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")

    df = clean(df)

    counts = df.groupby(ITEM_COL)[JUDGE_COL].nunique()
    incomplete = counts[counts < counts.max()]
    if len(incomplete):
        print(f"[warning] {len(incomplete)} item(s) have fewer than "
              f"{counts.max()} judge ratings (dropped rows / failed judging).")

    alpha_report = compute_all_alphas(df)
    print("\nKrippendorff's alpha per metric:")
    print(alpha_report.to_string(index=False))

    agg_df = aggregate_scores(df)
    print(f"\nAggregated per-item scores (first 5 of {len(agg_df)}):")
    print(agg_df.head().to_string(index=False))

    return alpha_report, agg_df


In [3]:
def get_output_dir() -> str:
    """
    Resolve a writable directory for outputs.
 
    Kaggle notebooks can only write to /kaggle/working/ -- anywhere else
    is typically read-only or gets wiped between sessions. This picks
    /kaggle/working if it exists (i.e. we're running on Kaggle),
    otherwise falls back to the current directory.
    """
    kaggle_dir = "/kaggle/working"
    if os.path.isdir(kaggle_dir):
        return kaggle_dir
    return "."
 
 
def save_and_link(df: pd.DataFrame, filename: str) -> str:
    """
    Save a DataFrame to CSV in the resolved output directory and, if
    running inside a Jupyter/IPython session (as Kaggle notebooks do),
    print a clickable download link right below the cell -- no need to
    commit the notebook or dig through the Output tab.
    """
    out_dir = get_output_dir()
    path = os.path.join(out_dir, filename)
    df.to_csv(path, index=False)
 
    try:
        from IPython.display import FileLink, display
        display(FileLink(path))
    except ImportError:
        pass  
 
    print(f"Saved: {path}")
    return path

In [4]:
if __name__ == "__main__":
    results = {}
    for name, repo_id in HF_DATASETS.items():
        raw_df = load_judged_dataset(repo_id)
        alpha_report, agg_df = analyze_dataset(name, raw_df)
        results[name] = {"alpha": alpha_report, "aggregated": agg_df}
 
        save_and_link(alpha_report, f"{name}_krippendorff_alpha.csv")
        save_and_link(agg_df, f"{name}_aggregated_scores.csv")
 
    print(f"\nDone. Files are in: {get_output_dir()}")

data/train-00000-of-00001.parquet:   0%|          | 0.00/623k [00:00<?, ?B/s]


exp10
[warning] 18 item(s) have fewer than 3 judge ratings (dropped rows / failed judging).

Krippendorff's alpha per metric:
                             metric level_of_measurement  krippendorff_alpha  n_items  n_judges
        recomputed_formatting_score              ordinal            0.802068      200         3
       recomputed_readability_score              ordinal           -0.153921      200         3
recomputed_formatting_checks_passed             interval            0.778831      200         3
       recomputed_readability_ratio                ratio           -0.029727      200         3

Aggregated per-item scores (first 5 of 200):
 item_id  recomputed_formatting_score_mean  recomputed_formatting_score_median  recomputed_formatting_score_std  recomputed_formatting_score_count  recomputed_readability_score_mean  recomputed_readability_score_median  recomputed_readability_score_std  recomputed_readability_score_count  recomputed_formatting_checks_passed_mean  recomputed_form

/kaggle/working/exp10_krippendorff_alpha.csv

Saved: /kaggle/working/exp10_krippendorff_alpha.csv


/kaggle/working/exp10_aggregated_scores.csv

...


gpt4.1
[warning] 9 item(s) have fewer than 3 judge ratings (dropped rows / failed judging).

Krippendorff's alpha per metric:
                             metric level_of_measurement  krippendorff_alpha  n_items  n_judges
        recomputed_formatting_score              ordinal            0.043795      200         3
       recomputed_readability_score              ordinal           -0.158912      200         3
recomputed_formatting_checks_passed             interval            0.140141      200         3
       recomputed_readability_ratio                ratio           -0.057778      200         3

Aggregated per-item scores (first 5 of 200):
 item_id  recomputed_formatting_score_mean  recomputed_formatting_score_median  recomputed_formatting_score_std  recomputed_formatting_score_count  recomputed_readability_score_mean  recomputed_readability_score_median  recomputed_readability_score_std  recomputed_readability_score_count  recomputed_formatting_checks_passed_mean  recomputed_form

/kaggle/working/gpt4.1_krippendorff_alpha.csv

Saved: /kaggle/working/gpt4.1_krippendorff_alpha.csv


/kaggle/working/gpt4.1_aggregated_scores.csv

Saved: /kaggle/working/gpt4.1_aggregated_scores.csv

Done. Files are in: /kaggle/working
